# Advanced Machine Learning Emotion Classification

This notebook trains and tunes advanced machine learning models (XGBoost, LightGBM, and SVM) on pre-extracted audio features to achieve a higher classification F1-score than Random Forest.

## Methodology
1. **Preprocessing**: Handle missing/invalid target labels, drop duplicates, and apply median imputation for missing feature cells.
2. **Feature Scaling**: Apply standard scaling (`StandardScaler`) to features.
3. **Tuning**: Tune hyperparameters for XGBoost, LightGBM, and SVM using **Optuna**.
4. **Evaluation**: Compare cross-validation performance and evaluate the best model on the unseen test set.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, f1_score, cohen_kappa_score
from sklearn.svm import SVC
try:
    import xgboost as xgb
except ImportError:
    %pip install xgboost -q
    import xgboost as xgb
try:
    import lightgbm as lgb
except ImportError:
    %pip install lightgbm -q
    import lightgbm as lgb
import optuna

In [12]:
_base = os.getcwd()
_project_root = os.path.dirname(_base) if os.path.basename(_base).lower() == "training" else _base
ALL_EMOTIONS_CSV = os.path.normpath(os.path.join(_project_root, "dataset", "all_emotions.csv"))

df = pd.read_csv(ALL_EMOTIONS_CSV)
print("Shape:", df.shape)

Shape: (57913, 49)


In [13]:
TARGET_COL = "label"
if TARGET_COL not in df.columns and "Label" in df.columns:
    TARGET_COL = "Label"

# Clean null targets and 'nan' strings
df_cleaned = df.dropna(subset=[TARGET_COL]).copy()
df_cleaned = df_cleaned[df_cleaned[TARGET_COL].astype(str).str.strip().str.lower() != "nan"]

FEATURE_COLS = ["F0_mean", "F0_std", "F0_range", "Energy_ mean", "Energy_ std", "ZCR_mean", "ZCR_std", "Spectral_centroid_mean", "Spectral_centroid_std", "Spectral_flux_mean", "MFCC_C0_mean", "MFCC_C1_mean", "MFCC_C2_mean", "MFCC_C3_mean", "MFCC_C5_mean", "MFCC_C7_mean", "MFCC_C10_mean", "MFCC_C0_std", "MFCC_C1_std", "MFCC_C2_std", "MFCC_C3_std", "MFCC_C5_std", "MFCC_C7_std", "Delta_MFCC_C0_std", "Delta_MFCC_C2_std", "Delta_MFCC_C3_std"]

# Impute missing cells in features with median
for col in FEATURE_COLS:
    s = pd.to_numeric(df_cleaned[col], errors="coerce")
    s = s.replace([np.inf, -np.inf], np.nan)
    med = s.median()
    if pd.isna(med):
        med = 0.0
    df_cleaned[col] = s.fillna(med)

X = df_cleaned[FEATURE_COLS].copy()
y = df_cleaned[TARGET_COL].astype(str).str.strip()

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Preprocessed X shape:", X.shape)
print("Classes:", le.classes_)

Preprocessed X shape: (54485, 26)
Classes: ['anger' 'disgust' 'fear' 'happy' 'neutral' 'sad']


In [14]:
RANDOM_STATE = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=RANDOM_STATE, stratify=y_encoded
)

# Apply standard scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape, "| Test shape:", X_test_scaled.shape)

Train shape: (43588, 26) | Test shape: (10897, 26)


### XGBoost Tuning

In [15]:
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'eval_metric': 'mlogloss'
    }
    
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_scaled, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
    )
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    preds = model.predict(X_val)
    return f1_score(y_val, preds, average='weighted')

optuna.logging.set_verbosity(optuna.logging.WARNING)
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=30, show_progress_bar=True)
print("Best XGBoost Val F1:", study_xgb.best_value)
print("Best XGBoost Params:", study_xgb.best_params)

Best trial: 25. Best value: 0.833458: 100%|██████████| 30/30 [15:45<00:00, 31.52s/it]

Best XGBoost Val F1: 0.8334582426563992
Best XGBoost Params: {'n_estimators': 468, 'max_depth': 10, 'learning_rate': 0.17549140891728818, 'subsample': 0.9690394070981359, 'colsample_bytree': 0.7725519831966194, 'gamma': 1.0492767129301485e-08}


### LightGBM Tuning

In [16]:
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 10, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'verbose': -1
    }
    
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_scaled, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
    )
    
    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])
    preds = model.predict(X_val)
    return f1_score(y_val, preds, average='weighted')

study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb, n_trials=30, show_progress_bar=True)
print("Best LightGBM Val F1:", study_lgb.best_value)
print("Best LightGBM Params:", study_lgb.best_params)

Best trial: 13. Best value: 0.842069: 100%|██████████| 30/30 [04:22<00:00,  8.74s/it]

Best LightGBM Val F1: 0.842069322599305
Best LightGBM Params: {'n_estimators': 499, 'max_depth': 11, 'num_leaves': 67, 'learning_rate': 0.24625126683753454, 'subsample': 0.6909971314141472, 'colsample_bytree': 0.7554088091076056}


### Support Vector Machine (SVM) Tuning

Note: Since SVM has $O(N^2)$ to $O(N^3)$ computational complexity, tuning it on all 43,000 training samples would take a very long time. We perform Optuna tuning on a representative stratified subset of 5,000 samples.

In [17]:
def objective_svm(trial):
    params = {
        'C': trial.suggest_float('C', 0.1, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 1e-4, 1.0, log=True),
        'kernel': 'rbf',
        'random_state': RANDOM_STATE
    }
    
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_scaled, y_train, test_size=0.2, train_size=5000, random_state=RANDOM_STATE, stratify=y_train
    )
    
    model = SVC(**params)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_val)
    return f1_score(y_val, preds, average='weighted')

study_svm = optuna.create_study(direction='maximize')
study_svm.optimize(objective_svm, n_trials=15, show_progress_bar=True)
print("Best SVM Val F1:", study_svm.best_value)
print("Best SVM Params:", study_svm.best_params)

Best trial: 13. Best value: 0.721882: 100%|██████████| 15/15 [00:33<00:00,  2.24s/it]

Best SVM Val F1: 0.721882409752954
Best SVM Params: {'C': 1.5055936534642158, 'gamma': 0.15084146148992994}


### Train and Evaluate the Best Final Model on Test Set

In [18]:
# Determine the best model type based on validation results
best_model_name = "XGBoost"
best_val_score = study_xgb.best_value
best_params = study_xgb.best_params

if study_lgb.best_value > best_val_score:
    best_model_name = "LightGBM"
    best_val_score = study_lgb.best_value
    best_params = study_lgb.best_params

if study_svm.best_value > best_val_score:
    best_model_name = "SVM"
    best_val_score = study_svm.best_value
    best_params = study_svm.best_params

print(f"Selected Best Model: {best_model_name} with Validation F1 of {best_val_score:.4f}")

if best_model_name == "XGBoost":
    final_model = xgb.XGBClassifier(**best_params, random_state=RANDOM_STATE, n_jobs=-1)
elif best_model_name == "LightGBM":
    final_model = lgb.LGBMClassifier(**best_params, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
else:
    final_model = SVC(**best_params, random_state=RANDOM_STATE)

# Fit on the full training set
final_model.fit(X_train_scaled, y_train)

# Evaluate on unseen test set
y_pred = final_model.predict(X_test_scaled)

print("\n=== Test Set Classification Report ===")
print(classification_report(y_test, y_pred, target_names=le.classes_))

f1w = f1_score(y_test, y_pred, average="weighted")
kappa = cohen_kappa_score(y_test, y_pred)
print(f"Final Test F1 (weighted): {f1w:.4f}")
print(f"Final Test Cohen Kappa:   {kappa:.4f}")

Selected Best Model: LightGBM with Validation F1 of 0.8421

=== Test Set Classification Report ===
              precision    recall  f1-score   support

       anger       0.94      0.90      0.92      1863
     disgust       0.83      0.85      0.84      1863
        fear       0.85      0.83      0.84      1863
       happy       0.82      0.83      0.82      1863
     neutral       0.79      0.84      0.81      1583
         sad       0.88      0.85      0.86      1862

    accuracy                           0.85     10897
   macro avg       0.85      0.85      0.85     10897
weighted avg       0.85      0.85      0.85     10897

Final Test F1 (weighted): 0.8506
Final Test Cohen Kappa:   0.8201
